# TEE Regression Exports
Carolina Carreira

This notebook contains code for performing the exports for the regressions. 

# Setup

In [53]:
import pandas as pd
import seaborn as sns
import matplotlib.pylab as plt
import surveyfields
import numpy as np
import os
import chardet   


pd.set_option('display.width', 400)
pd.set_option('display.max_columns', 10)

sns.set_theme()

ENCODING = 'utf-8-sig'
EXPORT_ENCODING = "UTF-8"
RESPONSE_ID = "ResponseId"
EXPORT_DIR = "./data/firststudy/"
PATH_EXCLUSION = EXPORT_DIR+"exclusions.csv"
PATH_DEMOGRAPHIC_QUAL = "export/demographic-recoded.csv"
PATH_WILLIGNESS = EXPORT_DIR+"willingness-recoded.csv"
PATH_QUALITATIVE = EXPORT_DIR+"qualitative_data.csv"

### Read data and some basic cleaning

* Read scored data
* Remove unneccessary columns
* Rename participant ids so they are unique between surveys

In [54]:
# Read data, drop unnecessary qualtrics columns
results = pd.read_csv('./export/anonymized_data.csv', encoding=ENCODING)
with open('./export/demographic-recoded.csv', 'rb') as f:
    result = chardet.detect(f.read())  # Read enough bytes to guess the encoding
    encoding = result['encoding']
    
demographics = pd.read_csv("export/demographic-recoded.csv", encoding=encoding)

def export_csv(pd, path, encoding):
  pd.to_csv(path, header=True, index=False, encoding=encoding)
  return 0

### Recode fields for regressions

This block recodes several of the background/experience fields into more
condensed versions more suitable for running regression analyses.
* Reduce number of levels in age, education, and gender
* Convert demographic specific things to True and False booleans (e.g. IoT device ownsership)

In [55]:
def aux_apply_map_clean(df, map, column):
  # applies map and keeps the column name
  df[column+'aux'] = df[column].apply(lambda x: map[x])
  df[column] = df[column+"aux"].copy()
  
  return df.drop(column+"aux", axis=1)

def recode_for_regression(df):

  # Recode age into 10 year buckets
  ageMap = {
    '18-24 years old': '18-24',
    '25-34 years old': '25-34',
    '35-44 years old': '35-44',
    '45-54 years old': '45+',
    '55-64 years old': '45+',
    '65+ years old': '45+',
    'Prefer not to say': 'Prefer not to say'
  }
  
  df['age_reduced'] = df['Age'].apply(lambda age: ageMap[age])
  df["Age"] = df["age_reduced"].copy()
  df = df.drop('age_reduced', axis=1)

  df['gender'] = df['Gender'].replace({
    'Female': 'Female NB or SD',
    'Prefer to self-describe': 'Female NB or SD',
    'Non-binary': 'Female NB or SD',
    "Non-binary / third gender": 'Female NB or SD'
}) 

  # Recode education into two buckets
  educationMap = {
    'Associates or technical degree': 'Bachelor or Graduate degree',
    'Bachelor’s degree': 'Bachelor or Graduate degree',
    'Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS etc.)': 'Bachelor or Graduate degree',
    'High school diploma or GED': 'High school or below',
    'Some high school or less': 'High school or below',
    'Other': 'Other',
    'Some college, but no degree': 'High school or below',
    'Prefer not to say': 'Prefer not to say'
  } 
  df['education_reduced'] = df['Education'].apply(lambda edu: educationMap[edu])
  df["Education"] = df["education_reduced"].copy()
  df = df.drop("education_reduced", axis=1) 

  # Recode Medical Job to True or False
  medicalExpMap = {
    'Yes; as a participant': True,
    'Yes; as a researcher': True,
    'No': False,
    'I’m not sure/Other': False
  }
  df['previous_medical_research'] = df['Demograhic-Medical'].apply(lambda x: medicalExpMap[x])

  # Recode Medical Experience to True or False
  medicalJobMap = {
    'Yes': True,
    'No, but I have been in the past': True,
    'No': False,
    'I’m not sure/Other': False
  }
  df['medical_job'] = df['Demograhic-Medical2n'].apply(lambda x: medicalJobMap[x])

  # Recode ownership of IoT to True or False
  IoTMap = {
    'Yes, and I use the smart features': True,
    'Yes, but I don’t use the smart features': True,
    'No': False,
    "No, but I have in the past": True,
    'I’m not sure/Other': False
  }
  df['IoT_ownership'] = df['Demograhic-IoT'].apply(lambda x: IoTMap[x])

  # Recode IoT automation to True or False
  IoTMap = {
    'Yes, I currently use home automation': True,
    'Yes, I have used home automation, but not currently': True,
    'No, I don’t own smart devices': False,
    "No, I do not use home automation, but I do own smart devices": False,
    'I’m not sure/Other': False
  }
  df['IoT_automation'] = df['Demograhic-IoT-2nd'].apply(lambda x: IoTMap[x])

  # Recode employment in IT field
  TechMap = {
    "Yes": True,
    "No": False,
    "No, but I have been in the past": True,
    "I’m not sure/Other": False
  }
  df['CS_Employment'] = df['Computing-M'].apply(lambda x: TechMap[x])

  # Recode education in CS 
  TechMap = {
    "Yes": True,
    "No": False,
    "I'm not sure/Other": False
  }
  df['CS_Education'] = df['ComputingEd-M'].apply(lambda x: TechMap[x])

  
  return df

demographics = recode_for_regression(demographics.copy())

def reduce_predictors(df):
  df['combined_IoT_Experience'] = df['IoT_ownership'] | df['IoT_automation']
  df['combined_Medical_Experience'] = df['previous_medical_research'] | df['medical_job']
  df['combined_CS_Experience'] = df['CS_Employment'] | df['CS_Education']

  return df
  
demographics = reduce_predictors(demographics.copy())

In [56]:
def recode_scenario(df):
  willingness = {
      'Definitely would': 'Definitely would',
      'Maybe would': 'Maybe would',
      'Would not ': 'Would not',
      'Not sure (Why?)': 'Would not'
    }
  #df = aux_apply_map_clean(df, willingness, "willingness")

    # Recode safe
    
  safe = {
      "Completely safe": "Completely safe",
      "Mostly safe": "Completely safe",
      "Somewhat safe": "Somewhat safe",
      "Not at all safe": "Not at all safe"
  }
  df = aux_apply_map_clean(df, safe, "safety_post")
  df = aux_apply_map_clean(df, safe, "safety_pre")

   
  scenario = {
    "Medical-Simple": "MedicalSimple",
    "Medical-Complex": "MedicalComplex",
    "IoT-Simple": "IoTSimple",
    "IoT-Complex": "IoTComplex"
  }

  scenario_complexity = {
    "Medical-Simple": "Simple",
    "Medical-Complex": "Complex",
    "IoT-Simple": "Simple",
    "IoT-Complex": "Complex"
  }

  scenario_scenario = {
    "Medical-Simple": "Medical",
    "Medical-Complex": "Medical",
    "IoT-Simple": "IoT",
    "IoT-Complex": "IoT"
  }

  df['scenario_complexity'] = df['scenario'].apply(lambda x: scenario_complexity[x])
  df['scenario_scenario'] = df['scenario'].apply(lambda x: scenario_scenario[x])
  df = aux_apply_map_clean(df, scenario, "scenario")
  return df


In [57]:
regression_demographics_columns = [
    'ResponseId',

    # demographic predictors
    'Age',
    'gender',
    "Education",
    # study specific predictors
    "previous_medical_research",
    "medical_job",
    "combined_IoT_Experience",
    "combined_Medical_Experience",

    "combined_CS_Experience"
]

regression_demographics_results_columns = [
    'ResponseId',
]
regression_demographics = pd.merge(demographics[regression_demographics_columns], results[regression_demographics_results_columns], on='ResponseId')
regression_demographics = regression_demographics.rename(columns={
    'Age': 'age',
    'Education': 'education',
    'gender':"gender"
})

""" regression_demographics['education'] = regression_demographics['education'].replace({
     'Have not completed high school': 'High school or below',
     'High school or equivalent': 'High school or below',
     'Other': 'High school or below'
}) """

# regression_demographics['Gender'] = regression_demographics['Gender'].replace({
#     'Female': 'Female NB or SD',
#     "Prefer not to say": 'Female NB or SD',
#     'Prefer to self-describe': 'Female NB or SD',
#     'Non-binary / third gender': 'Female NB or SD'
# })

" regression_demographics['education'] = regression_demographics['education'].replace({\n     'Have not completed high school': 'High school or below',\n     'High school or equivalent': 'High school or below',\n     'Other': 'High school or below'\n}) "

In [58]:
variables_to_melt = {
    "willingness": ["willingness-medical-merged","willingness-IoT-merged"],
    "safety_post": ["safety-medical-merged-post","safety-IoT-merged-post"],
    "safety_pre": ["safety-medical-merged-pre","safety-IoT-merged-pre"]
}


def melt_aux(df, final_df, new_colum, original_columns):
    med_df = pd.melt(df, id_vars=['ResponseId', 'SCENARIO_Med'], value_vars=[original_columns[0]],
                        var_name='Question', value_name=new_colum)
    med_df['scenario'] = med_df['SCENARIO_Med']
    med_df.drop(columns=['Question', 'SCENARIO_Med'], inplace=True)

    iot_df = pd.melt(df, id_vars=['ResponseId', 'SCENARIO_IoT'], value_vars=[original_columns[1]],
                        var_name='Question', value_name=new_colum)
    iot_df['scenario'] = iot_df['SCENARIO_IoT']
    iot_df.drop(columns=['Question', 'SCENARIO_IoT'], inplace=True)

    current_df = pd.concat([med_df, iot_df], ignore_index=True)
    
    if final_df.empty:
        final_df = current_df
    else:
        final_df = pd.merge(final_df, current_df, on=['ResponseId', 'scenario'], how='outer')
    return final_df


def aggregate_scenario(df):
    final_df = pd.DataFrame()
    

    for new_column_name in variables_to_melt:
        final_df = melt_aux(df, final_df,new_column_name, variables_to_melt[new_column_name])
        
    final_df.sort_values('ResponseId', inplace=True)
    final_df.reset_index(drop=True, inplace=True)

    return final_df

### Export for Regression about Scores

In [59]:
export_results_IoT = pd.merge(regression_demographics, results[["ResponseId"] + surveyfields.iotScoreFields], on='ResponseId')

export_results_Medical = pd.merge(regression_demographics, results[["ResponseId"] + surveyfields.medicalScoreFields], on='ResponseId')

all_results = pd.merge(regression_demographics, results[surveyfields.resultsScoreFields], on='ResponseId')
# Splitting the DataFrame into two based on scenario type
medical_df = all_results[['ResponseId', 'SCENARIO_Med',  'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10']].copy()
iot_df = all_results[['ResponseId', 'SCENARIO_IoT', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'I10']].copy()

# Adjusting the medical for the output structure
medical_df['SCENARIO'] = 'Medical'
medical_df['COMPLEXITY'] = medical_df['SCENARIO_Med'].apply(lambda x: 'Complex' if 'Complex' in x else 'Simple')
medical_df['A'] = medical_df['M1']
medical_df['B'] = medical_df['M2']
medical_df['C'] = medical_df['M3']
medical_df['D'] = medical_df['M4']
medical_df['E'] = medical_df['M5']
medical_df['F'] = medical_df['M6']
medical_df['G'] = medical_df['M7']
medical_df['H'] = medical_df['M8']
medical_df['I'] = medical_df['M9']
medical_df['J'] = medical_df['M10']
medical_df.drop(['SCENARIO_Med', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10'], axis=1, inplace=True)

# Adjusting the iot for the output structure
iot_df['SCENARIO'] = 'IoT'
iot_df['COMPLEXITY'] = iot_df['SCENARIO_IoT'].apply(lambda x: 'Complex' if 'Complex' in x else 'Simple')
iot_df['A'] = iot_df['I1']
iot_df['B'] = iot_df['I2']
iot_df['C'] = iot_df['I3']
iot_df['D'] = iot_df['I4']
iot_df['E'] = iot_df['I5']
iot_df['F'] = iot_df['I6']
iot_df['G'] = iot_df['I7']
iot_df['H'] = iot_df['I8']
iot_df['I'] = iot_df['I9']
iot_df['J'] = iot_df['I10']
iot_df.drop(['SCENARIO_IoT', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'I10'], axis=1, inplace=True)

# Concatenate the two  
scores_regression = pd.concat([medical_df, iot_df], ignore_index=True)
scores_regression = pd.merge(scores_regression, regression_demographics, on='ResponseId')
scores_regression = pd.merge(scores_regression, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId')

# sort and reset index
scores_regression.sort_values(by='ResponseId', inplace=True)
scores_regression.reset_index(drop=True, inplace=True)

export_csv(export_results_IoT, EXPORT_DIR+ 'regression_IoT.csv', "UTF-8")
export_csv(export_results_Medical, EXPORT_DIR+ 'regression_Medical.csv', "UTF-8")
scores_regression['scenario'] = scores_regression['SCENARIO'].astype(str) + scores_regression['COMPLEXITY'].astype(str)
export_csv(scores_regression, EXPORT_DIR+ 'scores_regression.csv', "UTF-8")

0

### Export for Willigness Regression

In [60]:
# rename "How likely would you be to purchase the voice assistant/light bulb?" to willingness-IoT-merged
# rename "How likely would you be willing to participate in the medical study?" to willingness-medical-merged
results = results.rename(columns={"How likely would you be to purchase the voice assistant/light bulb?": 'willingness-IoT-merged', "How likely would you be willing to participate in the medical study?": 'willingness-medical-merged'})
## POS-TEE CONCEPTS
# rename "How safe do you think data would be when it is protected by a TEE? (IoT, post-TEE concepts)" to safety-IoT-merged
# rename "How safe do you think data would be when it is protected by a TEE? (medical, post-TEE concepts)" to safety-medical-merged

results = results.rename(columns={"How safe do you think data would be when it is protected by a TEE? (IoT, post-TEE concepts)": 'safety-IoT-merged-post', "How safe do you think data would be when it is protected by a TEE? (medical, post-TEE concepts)": 'safety-medical-merged-post'})

## PRE-TEE CONCEPTS
# rename "How safe do you think data would be when it is protected by a TEE? (IoT, post-TEE concepts)" to safety-IoT-merged
# rename "How safe do you think data would be when it is protected by a TEE? (medical, post-TEE concepts)" to safety-medical-merged

results = results.rename(columns={"How safe do you think data would be when it is protected by a TEE? (IoT, initial)": 'safety-IoT-merged-pre', "How safe do you think data would be when it is protected by a TEE? (medical, initial)": 'safety-medical-merged-pre'})


In [61]:

export_results_IoT = pd.merge(regression_demographics, results[["ResponseId"] + ['SCENARIO_IoT', 'willingness-IoT-merged']], on='ResponseId')
export_results_IoT = pd.merge(export_results_IoT, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId')

export_results_Medical = pd.merge(regression_demographics, results[["ResponseId"] + ['SCENARIO_Med', 'willingness-medical-merged']], on='ResponseId')
export_results_Medical = pd.merge(export_results_Medical, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId')


### Export for Trust Regression

In [62]:
export_results_IoT_post = pd.merge(regression_demographics, results[["ResponseId", 'SCENARIO_IoT', 'safety-IoT-merged-post']], on='ResponseId', how='outer')
export_results_IoT_post = pd.merge(export_results_IoT_post, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId', how='outer')

export_results_Medical_post = pd.merge(regression_demographics, results[["ResponseId",'SCENARIO_Med', 'safety-medical-merged-post']], 
on='ResponseId', how='outer')
export_results_Medical_post = pd.merge(export_results_Medical_post, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId', how='outer')

export_results_IoT_pre = pd.merge(regression_demographics, results[["ResponseId"] + ['SCENARIO_IoT', 'safety-IoT-merged-pre']], on='ResponseId')
export_results_IoT_pre = pd.merge(export_results_IoT_pre, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId')

export_results_Medical_pre = pd.merge(regression_demographics, results[["ResponseId"] + ['SCENARIO_Med', 'safety-medical-merged-pre']], on='ResponseId')
export_results_Medical_pre = pd.merge(export_results_Medical_pre, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId')

trust_all_post = pd.merge(export_results_IoT_post, export_results_Medical_post, on='ResponseId', how='left')
#print(export_results_Medical_post.columns)
trust_all_pre = pd.merge(export_results_IoT_pre, export_results_Medical_pre, on='ResponseId', how='left')
trust_all = pd.merge(trust_all_post, trust_all_pre, on='ResponseId', how='left')
willingness_all = pd.merge(export_results_IoT, export_results_Medical, on='ResponseId', how='left')
all = pd.merge(trust_all, willingness_all, on='ResponseId', how='left')

all = aggregate_scenario(all)

all = recode_scenario(all)

all = pd.merge(all, all_results[['ResponseId', "EXPLN_S1", "EXPLN_S2", "EXPLN_S3"]], on='ResponseId')
all = pd.merge(all,regression_demographics, on='ResponseId')
print(all.columns)
export_csv(all, EXPORT_DIR+ 'regression_willall.csv', "UTF-8")




Index(['ResponseId', 'willingness', 'scenario', 'safety_post', 'safety_pre', 'scenario_complexity', 'scenario_scenario', 'EXPLN_S1', 'EXPLN_S2', 'EXPLN_S3', 'age', 'gender', 'education', 'previous_medical_research', 'medical_job', 'combined_IoT_Experience', 'combined_Medical_Experience', 'combined_CS_Experience'], dtype='object')


0